# Feasibility, certificates and information

Mathematical certificates answer precise objective questions. Feasible domains, numerical residuals, held-out scores and computational budgets answer different questions and should be inspected together.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
ROOT=Path.cwd()
if ROOT.name == "notebooks": ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
def read(name): return json.loads((ROOT/name).read_text())


## A live domain point

In [2]:
def perspective_feasible(x,y): return y>0 and x*x/y <= x-2*y*y+5
assert perspective_feasible(1.,1.)
assert not perspective_feasible(1.,0.)
assert not perspective_feasible(3.,3.)
witnesses=read("results/reference/geometry/metrics.json")
pd.DataFrame([dict(domain=r["domain"],witness=r["witness"],violation=r["maximum_violation"]) for r in witnesses["rows"]])

,domain,witness,violation
0,simplex,"[0.29999999998124854, 0.399999999948868]",0.0
1,nonnegative_order,"[1.0000000000373959, 0.4999999999953881]",0.0
2,perspective_quadratic,"[0.9999999999988284, 0.9999999999934508]",0.0
3,concave_sqrt_hypograph,"[0.9999999999986605, -5.997745135156002e-12, 0...",0.0


## Inspect a projected feasible dual

In [3]:
from optimization_lab.primal_dual import certificate
b=np.random.default_rng(3).normal(size=(5,6))
cert=certificate(b,b,np.full((2,5,6),1e8),.2)
assert cert["dual_feasibility"] <= 1e-12
assert cert["gap"] >= -1e-12
cert

{'primal': 12.25679359944878,
 'dual': 0.0,
 'gap': 12.25679359944878,
 'dual_feasibility': 0.0}

## Exact route interval

In [4]:
from optimization_lab.routing import integer_one_tree,tour_length
data=np.load(ROOT/"assets/source_data/routing/berlin52.npz",allow_pickle=False)
bound=integer_one_tree(data["distance"],data["one_tree_potentials"])["bound"]
upper=tour_length(data["distance"],data["acs_2opt_seed0_tour"])
assert bound <= 7542 <= upper
print("Exact integer bounds:",bound,"<= optimum <=",upper)

Exact integer bounds: 7542 <= optimum <= 7542


## Compare evidence roles

In [5]:
audit=read("docs/extensions_validation.json")
assert audit["status"]=="passed"
print("72 feasible image solves and 96 real-data search candidates verified")
print("Open docs/explorer.html locally to inspect eight interactive concept groups. A heuristic route is an upper bound; validation selection is not a numerical optimality certificate.")

72 feasible image solves and 96 real-data search candidates verified
Open docs/explorer.html locally to inspect eight interactive concept groups. A heuristic route is an upper bound; validation selection is not a numerical optimality certificate.


## Exercise and worked interpretation

Change the modeling or budget assumption and identify which claim survives. A feasible TV dual remains a lower bound for its stated objective; it does not guarantee improved PSNR. A validation winner remains the winner of its declared candidates; a larger search can still overfit selection. A route certificate uses the exact integer metric, while a visual map is only a display. Keep these claims separate when interpreting the result.